# EduMentor AI — Module 2: ML Model Training
### Dropout Risk Predictor

This notebook:
1. Loads clean_data.csv
2. Splits into train/test sets
3. Trains Logistic Regression and Random Forest
4. Compares models using F1-Macro, ROC-AUC, Confusion Matrix
5. Selects and saves the best model as risk_model.joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✓')

## 1. Load Clean Dataset

In [ ]:
df = pd.read_csv('../data/clean_data.csv')
print(f'Loaded: {df.shape[0]} students, {df.shape[1]} columns')
print(f'\nTarget distribution (at_risk):')
print(df['at_risk'].value_counts())
print(f'\nImbalance ratio: {df["at_risk"].mean():.1%} positive (at risk)')

## 2. Prepare Features & Split

In [ ]:
FEATURE_COLS = [
    'age', 'gender_encoded', 'edu_level_encoded', 'num_prev_attempts',
    'studied_credits', 'disability_encoded', 'region_encoded',
    'avg_score', 'days_active', 'forum_posts', 'resource_clicks',
    'assignment_submissions', 'video_views', 'quiz_attempts'
]

X = df[FEATURE_COLS]
y = df['at_risk']

# Stratified split to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nTrain class balance: {y_train.mean():.1%} at risk')
print(f'Test class balance:  {y_test.mean():.1%} at risk')

## 3. Model A — Logistic Regression

In [ ]:
# Pipeline: scaling → logistic regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Handle imbalanced classes
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

# Metrics
lr_f1 = f1_score(y_test, lr_pred, average='macro')
lr_auc = roc_auc_score(y_test, lr_proba)

print('=== Logistic Regression Results ===')
print(f'F1-Macro:  {lr_f1:.4f}')
print(f'ROC-AUC:   {lr_auc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Low Risk', 'High Risk']))

## 4. Model B — Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

# Metrics
rf_f1 = f1_score(y_test, rf_pred, average='macro')
rf_auc = roc_auc_score(y_test, rf_proba)

print('=== Random Forest Results ===')
print(f'F1-Macro:  {rf_f1:.4f}')
print(f'ROC-AUC:   {rf_auc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=['Low Risk', 'High Risk']))

## 5. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')

# --- Confusion Matrices ---
for ax, pred, title in zip(
    axes[:2],
    [lr_pred, rf_pred],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

# --- ROC Curves ---
ax = axes[2]
for proba, label, color in [
    (lr_proba, f'Logistic Reg (AUC={lr_auc:.3f})', '#5C6BC0'),
    (rf_proba, f'Random Forest (AUC={rf_auc:.3f})', '#26A69A')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=label, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'F1-Macro': [lr_f1, rf_f1],
    'ROC-AUC': [lr_auc, rf_auc],
})
results_df = results_df.round(4)
print('=== Final Model Comparison ===')
print(results_df.to_string(index=False))

## 6. Feature Importance (Random Forest)

In [ ]:
importances = rf_pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh', color='#26A69A', edgecolor='white')
plt.title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 features:')
print(feat_imp.sort_values(ascending=False).head())

## 7. Cross-Validation & Select Best Model

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='f1_macro')
rf_cv = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='f1_macro')

print('=== 5-Fold Cross-Validation (F1-Macro) ===')
print(f'Logistic Regression: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}')
print(f'Random Forest:       {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

# Select best
best_model_name = 'Random Forest' if rf_cv.mean() >= lr_cv.mean() else 'Logistic Regression'
best_pipeline = rf_pipeline if rf_cv.mean() >= lr_cv.mean() else lr_pipeline

print(f'\n🏆 Best Model: {best_model_name}')

## 8. Save Best Model

In [ ]:
import os
os.makedirs('../backend/models', exist_ok=True)

model_path = '../backend/models/risk_model.joblib'
joblib.dump(best_pipeline, model_path)

print(f'✅ Model saved to: {model_path}')
print(f'   Model type: {best_model_name}')
print(f'   Best CV F1-Macro: {max(rf_cv.mean(), lr_cv.mean()):.4f}')

# Verify
loaded = joblib.load(model_path)
test_pred = loaded.predict(X_test[:3])
print(f'   Verification predictions: {test_pred} ✓')